# Predictive AI Evaluation Challenge — Metadata-only Latent-Factor Model (Colab)

This notebook clones the competition repo, downloads the public HuggingFace response parquets, trains a metadata-only PyTorch latent-factor model, and runs the official-like validation harness across 3 seeds.

**Statistical form**
$$\eta_{m,b,c} = \mu + a_m + b_{b,c} + \frac{u_m \cdot v_{b,c}}{\sqrt{k}}$$
$$p_{m,b,c} = \sigma(\eta_{m,b,c})$$

The model intentionally **does not see `item_content`**. Headline baseline to beat: a no-cross-term logistic regression with mean log-likelihood $\approx -0.5224$.

**Recommended runtime**: A100 (Runtime → Change runtime type → GPU → A100). L4/T4 also work — the model is small and the bottleneck is data movement.

## 1. Environment + GPU info

In [ ]:
import os, sys, subprocess, json, time, shutil
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'GPU memory: free={free/1e9:.2f}GB total={total/1e9:.2f}GB')
except ImportError:
    print('torch is not installed yet — will install in next cell.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Install dependencies

Colab already ships `torch`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`. We only need to ensure `huggingface_hub` and `datasets` are present for the data download. We also install `tqdm` (already there but pinned for safety).

In [ ]:
%pip install -q --upgrade huggingface_hub datasets pyarrow pandas numpy scikit-learn tqdm

## 3. Clone the competition repo

The repo bundles `validation_harness/`, `starting_kit/Model_Info/model_info.csv`, `starting_kit/benchmark_info/benchmark_info.csv`, and `Google_Collab_harness/` (this folder). It does **not** include the response parquets — those come from HuggingFace in the next step.

In [ ]:
REPO_URL = 'https://github.com/bwathomas/Prediction-Competition-321M.git'
REPO_DIR = Path('/content/Prediction-Competition-321M').resolve()
if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists, pulling latest …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

for sub in ['validation_harness', 'starting_kit/Model_Info', 'starting_kit/benchmark_info', 'Google_Collab_harness']:
    p = REPO_DIR / sub
    print(f'  {sub:40s} {"OK" if p.exists() else "MISSING"}')

GCH = REPO_DIR / 'Google_Collab_harness'
if str(GCH) not in sys.path:
    sys.path.insert(0, str(GCH))
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 4. Download the response parquets from HuggingFace

We download all `*.parquet` files from `aims-foundations/measurement-db` except the `*_traces.parquet` ones (different schema, not used). Total ≈ 1.5 GB.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = 'aims-foundations/measurement-db'
DATA_DIR = REPO_DIR / 'starting_kit' / 'Data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type='dataset')
wanted = [
    f for f in files
    if f.endswith('.parquet') and not f.endswith('_traces.parquet')
]
print(f'{len(wanted)} parquets to download')

for f in wanted:
    out = DATA_DIR / Path(f).name
    if out.exists() and out.stat().st_size > 0:
        continue
    print(f'  downloading {f} …', flush=True)
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset', local_dir=str(DATA_DIR), local_dir_use_symlinks=False)
    if Path(p).resolve() != out.resolve():
        try:
            shutil.copy2(p, out)
        except shutil.SameFileError:
            pass
print('done. files:')
subprocess.run(['ls', '-lh', str(DATA_DIR)], check=False)

## 5. Build the official item-cold-start split

Reuses `validation_harness/scripts/prepare_split.py` so the validation we report is exactly the official-like protocol.

In [ ]:
HARNESS_DIR = REPO_DIR / 'validation_harness'
SPLITS_DIR = HARNESS_DIR / 'splits' / 'v1'
if not (SPLITS_DIR / 'train.parquet').exists():
    subprocess.check_call([
        sys.executable,
        str(HARNESS_DIR / 'scripts' / 'prepare_split.py'),
        '--data-dir', str(DATA_DIR),
        '--out-dir',  str(SPLITS_DIR),
        '--val-fraction', '0.10',
        '--seed', '0',
    ])
else:
    print(f'Reusing existing split at {SPLITS_DIR}')
subprocess.run(['ls', '-lh', str(SPLITS_DIR)], check=False)

## 6. Train the latent-factor model + run official-like validation

This wraps everything: fit preprocessor on TRAIN ONLY, aggregate by (model, benchmark, condition) cells, train with AdamW + AMP + early stopping, package a submission folder, and run `validation_harness/scripts/run_validation.py` across seeds 0/1/2.

Defaults: `latent_dim=16`, `hidden_dim=256`, `num_layers=2`, `dropout=0.1`, `batch_size=65536`, `epochs=30`, `patience=5`.

**Live progress reporting**

- A live `tqdm` bar tracks the planned **total training steps** (= `epochs * steps_per_epoch`) with continuously updated **% done** and **ETA**. The bar's postfix shows the current `epoch`, the current batch `loss`, and the best validation log-likelihood seen so far (`best_val`).
- Every **10 optimizer steps** (`--log-every-steps 10`) a one-line summary is printed: `step S/Total (X% done with training, time estimated is Y)  epoch=…  loss=…  elapsed=…`.
- At the end of every **epoch** a per-epoch summary is logged (`train_loss`, `val_ll`, `val_brier`, `val_auc`, `epoch_seconds`, `eta`, current best with epoch). A trailing `*` marks an improvement.

In [ ]:
import os, subprocess

OUTPUT_DIR = REPO_DIR / 'outputs' / 'latent_factor'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', str(GCH / 'run_latent_factor_colab.py'),
    '--data-dir',                str(DATA_DIR),
    '--splits-dir',              str(SPLITS_DIR),
    '--model-info-csv',          str(REPO_DIR / 'starting_kit' / 'Model_Info' / 'model_info.csv'),
    '--benchmark-info-csv',      str(REPO_DIR / 'starting_kit' / 'benchmark_info' / 'benchmark_info.csv'),
    '--validation-harness-dir',  str(HARNESS_DIR),
    '--output-dir',              str(OUTPUT_DIR),
    '--latent-dim', '16',
    '--hidden-dim', '256',
    '--num-layers', '2',
    '--dropout',    '0.1',
    '--weight-decay','1e-4',
    '--batch-size', '65536',
    '--epochs',     '30',
    '--patience',   '5',
    # --- live progress: per-step loss + tqdm-style "X% done, ETA Y" bar ---
    '--log-every-steps', '10',     # one-line training-step log every 10 optimizer steps
    '--progress-bar',              # tqdm bar over total training steps with live ETA
    # --- official-like validation across 3 seeds ---
    '--official-seeds', '0', '1', '2',
    '--official-n', '5000',
    '--official-k', '5',
    '--logistic-baseline-ll', '-0.5224',
]


def _run_streaming(argv):
    """Run a child process and stream its stdout/stderr into the cell.

    Plain ``subprocess.run(cmd)`` lets the child write to the kernel's raw
    stdout/stderr file descriptors, which IPython does NOT forward to the
    cell display in most Colab/Jupyter setups -- so the cell looks empty
    even though the script printed plenty. We capture stdout (with stderr
    merged in) and re-print line-by-line so IPython's OutStream picks it up.
    """
    print('Running:', ' '.join(argv), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    proc = subprocess.Popen(
        argv,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge so order is preserved
        bufsize=1,                 # line-buffered
        text=True,
        env=env,
    )
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    finally:
        proc.stdout.close()
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, argv)


_run_streaming(cmd)

## 7. (Optional) sequential hyperparameter sweep with full progress

Runs the same script with `--sweep`, which samples `--sweep-budget` configs from a 7-dimensional grid:

| dim | values |
| --- | --- |
| `latent_dim`   | 4, 8, 16, 32 |
| `hidden_dim`   | 128, 256 |
| `dropout`      | 0.05, 0.1, 0.2 |
| `weight_decay` | 1e-4, 1e-3 |
| `lr`           | 1e-3, 3e-3 |
| `id_emb_l2`    | 1e-4, 1e-3 |
| `patience`     | 5, 10 |

(`--sweep-mode full` would walk the entire 384-config grid; `random` picks a uniform subset of size `--sweep-budget`.)

**Why sequential here?** With `--parallel-runs 1` we get the *exact same* per-step progress as section 6 for **every** sweep run, one at a time. After each run finishes the next one starts and prints its own tqdm bar + per-step loss, so the cell output reads like a clean diary of "run 1 done → starting run 2 → ...". This is much easier to monitor than 8 interleaved tqdm bars on the same GPU. The shared preprocessor + tensor cache (built once in section 6) keeps per-run startup cheap.

If you'd rather trade observability for raw throughput, set `--parallel-runs 0` (auto) or e.g. `--parallel-runs 8` and the script will dispatch the configs through `ProcessPoolExecutor` with the `spawn` start method (tqdm bars are auto-disabled in that mode to avoid mangled output, but per-step + per-epoch text logs still go through).

In [ ]:
sweep_cmd = cmd + [
    '--sweep',
    '--sweep-mode', 'random',
    '--sweep-budget', '24',
    # Sequential: one process at a time so each run prints a clean tqdm bar +
    # per-10-step loss line, then we "pick up" the next run when it finishes.
    # Bump to e.g. 8 for parallel throughput (per-step bar is auto-disabled then).
    '--parallel-runs', '1',
    '--log-every-steps', '10',
    '--progress-bar',
    '--amp',
]
# Reuse the streaming helper from cell 12 so all subprocess output reaches
# the cell instead of disappearing into the kernel's raw stdout/stderr.
_run_streaming(sweep_cmd)

## 8. Read the metrics + interpret results

In [ ]:
import pandas as pd, json
summary = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
print(json.dumps(summary, indent=2))

print('\nbaseline_comparison.csv:')
print(pd.read_csv(OUTPUT_DIR / 'baseline_comparison.csv').to_string(index=False))

print('\nofficial_seeds.csv:')
print(pd.read_csv(OUTPUT_DIR / 'official_seeds.csv').to_string(index=False))

if (OUTPUT_DIR / 'runs.csv').exists():
    print('\nruns.csv (top 10 by final_val_log_likelihood):')
    runs = pd.read_csv(OUTPUT_DIR / 'runs.csv')
    cols = ['run_idx', 'latent_dim', 'weight_decay', 'dropout', 'best_epoch',
            'final_train_log_likelihood', 'final_val_log_likelihood', 'wall_seconds']
    cols = [c for c in cols if c in runs.columns]
    print(runs[cols].sort_values('final_val_log_likelihood', ascending=False).head(10).to_string(index=False))

off_mean = summary.get('official_mean_log_likelihood')
imp = summary.get('improvement_vs_logistic_baseline')
if off_mean is not None and imp is not None:
    verdict = 'BEATS' if (imp or 0) > 0 else 'TRAILS'
    print(f'\nOfficial-like mean LL = {off_mean:+.4f}    {verdict} the logistic baseline by {imp:+.4f}')

## 9. Download the best model

This cell produces **two** zip archives and triggers browser downloads via `google.colab.files`:

1. **`codabench_submission.zip`** — slim, **upload-ready for Codabench**. Files at the top level of the zip:
   - `model.py`                  — runtime entry point with `predict()`
   - `latent_factor_pytorch.py`  — model + preprocessor + inference classes
   - `best_model.pt`             — full bundle (`{state_dict, config_dict}`)
   - `preprocessor.pkl`          — fitted preprocessor (vocabs, scalers, lookups)
   - `model_info.csv`            — baked metadata (read by the preprocessor)
   - `benchmark_info.csv`        — baked metadata (read by the preprocessor)

   This is the file you upload to Codabench's "Submit / View Results" page. See section 10 for the submission walkthrough.

2. **`latent_factor_submission.zip`** — full bundle for archival / reproducibility, contains everything in (1) **plus** `weights.pt`, `metrics.json`, `runs.csv`, `baseline_comparison.csv`, `official_seeds.csv`, `reproduce.json`, `reproduce.sh`, `reproduce.bat`. Useful if you want to reload the model elsewhere or audit the run later:

   ```python
   from latent_factor_pytorch import load_artifacts
   model, preprocessor, config = load_artifacts('latent_factor_submission/')
   ```

Outside Colab the cell just prints absolute paths so you can copy the zips manually.

In [ ]:
import zipfile
from pathlib import Path

# Files that MUST live at the top level of the Codabench-bound zip so the
# platform's loader can find model.py and the model's runtime deps.
CODABENCH_REQUIRED = [
    'model.py',                  # entry point with predict()
    'latent_factor_pytorch.py',  # model + preprocessor + inference classes
    'best_model.pt',             # full bundle: state_dict + config_dict
    'preprocessor.pkl',          # fitted preprocessor (vocabs, scalers, lookups)
    'model_info.csv',            # baked metadata read at load time
    'benchmark_info.csv',        # baked metadata read at load time
]

# Extra files included in the archival bundle but NOT in the Codabench zip.
ARCHIVAL_EXTRAS = [
    'weights.pt',
    'metrics.json',
    'baseline_comparison.csv',
    'official_seeds.csv',
    'runs.csv',
    'reproduce.json',
    'reproduce.sh',
    'reproduce.bat',
]

# Resolve OUTPUT_DIR if it wasn't already defined (e.g. running this cell standalone)
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path('/content/Prediction-Competition-321M/outputs/latent_factor').resolve()

OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
SUBMISSION_DIR = OUTPUT_DIR / 'submission'
CODABENCH_ZIP = OUTPUT_DIR / 'codabench_submission.zip'
ARCHIVE_ZIP = OUTPUT_DIR / 'latent_factor_submission.zip'

print(f'Bundling artifacts under: {OUTPUT_DIR}')
if not OUTPUT_DIR.exists():
    raise SystemExit(f'OUTPUT_DIR does not exist: {OUTPUT_DIR}. Run section 6 first.')
if not SUBMISSION_DIR.exists():
    raise SystemExit(
        f'submission/ folder missing under {OUTPUT_DIR}. '
        f'It is built by run_latent_factor_colab.py at the end of section 6.'
    )

# ---------------------------------------------------------------------------
# 1. Codabench-ready zip: files at the TOP LEVEL of the archive.
#    Codabench unzips the upload directly, so model.py must be at the root.
# ---------------------------------------------------------------------------
print(f'\n[1/2] Building Codabench-ready zip -> {CODABENCH_ZIP}')
missing = []
with zipfile.ZipFile(CODABENCH_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in CODABENCH_REQUIRED:
        src = SUBMISSION_DIR / fname
        if not src.exists():
            missing.append(fname)
            continue
        zf.write(src, arcname=fname)  # top-level: NO wrapper folder
        print(f'    + {fname}  ({src.stat().st_size / 1024:.1f} KB)')
if missing:
    raise SystemExit(
        f'Codabench bundle missing required files: {missing}. '
        f'Re-run section 6 to rebuild submission/.'
    )
print(f'    => {CODABENCH_ZIP.name}  ({CODABENCH_ZIP.stat().st_size / 1e6:.2f} MB)')

# ---------------------------------------------------------------------------
# 2. Archival bundle: same files + reproducibility manifest, kept under a
#    wrapper folder for safe extraction.
# ---------------------------------------------------------------------------
print(f'\n[2/2] Building archival bundle -> {ARCHIVE_ZIP}')
with zipfile.ZipFile(ARCHIVE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    bundled, skipped = [], []
    for fname in CODABENCH_REQUIRED + ARCHIVAL_EXTRAS:
        src = OUTPUT_DIR / fname
        if not src.exists():
            src = SUBMISSION_DIR / fname  # some live only inside submission/
        if src.exists():
            zf.write(src, arcname=f'latent_factor_submission/{fname}')
            bundled.append(fname)
        else:
            skipped.append(fname)
    # Also tuck the full submission/ tree in for auditing.
    for p in SUBMISSION_DIR.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:
            arc = Path('latent_factor_submission/submission') / p.relative_to(SUBMISSION_DIR)
            zf.write(p, arcname=str(arc).replace('\\', '/'))
print(f'    bundled : {bundled}')
if skipped:
    print(f'    skipped : {skipped}  (not produced; ok if section 6 ran without --sweep)')
print(f'    => {ARCHIVE_ZIP.name}  ({ARCHIVE_ZIP.stat().st_size / 1e6:.2f} MB)')

# ---------------------------------------------------------------------------
# 3. Trigger browser downloads in Colab; gracefully degrade elsewhere.
# ---------------------------------------------------------------------------
try:
    from google.colab import files as _colab_files  # type: ignore
    print(f'\nTriggering download of {CODABENCH_ZIP.name} (Codabench-ready) ...')
    _colab_files.download(str(CODABENCH_ZIP))
    print(f'Triggering download of {ARCHIVE_ZIP.name} (archival) ...')
    _colab_files.download(str(ARCHIVE_ZIP))
except ImportError:
    print('\nNot running in Colab — no browser download triggered.')
    print(f'  Codabench zip: {CODABENCH_ZIP}')
    print(f'  Archive  zip: {ARCHIVE_ZIP}')
except Exception as e:
    print(f'\ngoogle.colab.files.download failed: {type(e).__name__}: {e}')
    print(f'Manual fallback paths:')
    print(f'  Codabench zip: {CODABENCH_ZIP}')
    print(f'  Archive  zip: {ARCHIVE_ZIP}')

## 10. Try the model in the actual competition

The downloaded `codabench_submission.zip` is the file you upload to Codabench. Concrete steps:

1. **Verify the bundle locally first** (next cell) — it imports the bundled `model.py`, calls `predict()` on a few synthetic inputs, and confirms each call returns a finite probability in `[0, 1]`. Catches >90% of "the bundle is broken" issues before you spend a daily submission slot. Takes <5 seconds.
2. **Upload to Codabench**. Open the competition page → "Submit / View Results" tab → "Submit a new entry" → select `codabench_submission.zip` (the file your browser downloaded in section 9, NOT the archival one). Codabench unzips it and looks for `model.py` at the top level (which is exactly what we shipped).
3. **Wait for scoring**. The platform materializes a hidden test slice (currently 5000 items, K=5 adaptive labels per category) and calls `predict()` once per `(subject, item)` pair. Module-level init runs once at container start and loads the bundled `best_model.pt` + `preprocessor.pkl` into memory; per-call latency is microseconds because the model is cached by `(benchmark, condition, model_name)`.
4. **Watch the leaderboard** for the submission's mean log-likelihood. The headline reference number from the existing logistic baseline is `−0.5224`. The number reported in `baseline_comparison.csv` from section 8 is what to expect on the public eval; the hidden test slice will be similar but not identical because it samples different items.

**Constraints to remember** (from `starting_kit/README.md`):

- Up to **50 scored submissions per team per UTC day**.
- The container has **no outbound internet**. Our submission is fully self-contained — no HuggingFace hub calls, no API keys, no external data.
- Module-level code (loading weights, building the preprocessor) runs **once**. Per-call work happens inside `predict()` and is dominated by a cache hit.
- We do **not** ship a `models.txt` (no HF repo dependency) so the platform routes us to the smallest available GPU tier (or CPU). That is fine — the model is tiny.
- We do **not** ship a `labeling.py`. Per the starter kit, that means the platform falls back to its default per-category random labeling. Our `predict()` ignores the `labeled` argument anyway (the model is item-content-independent), so this has no effect on our outputs.

In [ ]:
# Verify the codabench_submission.zip end-to-end BEFORE uploading.
# Unzips to a fresh temp dir, imports model.py from there, and runs predict()
# on a few representative inputs. If anything is wrong (missing file, version
# skew, broken pickle) you find out here -- not after a wasted Codabench slot.
import importlib, importlib.util, sys, tempfile, zipfile, time, math
from pathlib import Path

assert CODABENCH_ZIP.exists(), f'Run section 9 first — {CODABENCH_ZIP} not found.'

with tempfile.TemporaryDirectory() as td:
    extract_dir = Path(td) / 'submission'
    extract_dir.mkdir()
    print(f'Unzipping {CODABENCH_ZIP.name} -> {extract_dir} ...')
    with zipfile.ZipFile(CODABENCH_ZIP, 'r') as zf:
        zf.extractall(extract_dir)
    extracted = sorted(p.name for p in extract_dir.iterdir())
    print(f'  extracted files: {extracted}')

    # Make sure we import the *bundled* model.py, not any cached copy.
    for mod_name in ('model', 'latent_factor_pytorch'):
        sys.modules.pop(mod_name, None)
    sys.path.insert(0, str(extract_dir))
    try:
        t_init = time.perf_counter()
        spec = importlib.util.spec_from_file_location('model', extract_dir / 'model.py')
        model_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(model_mod)
        init_dt = time.perf_counter() - t_init
        print(f'  module init OK in {init_dt*1000:.1f} ms (loads weights + preprocessor)')

        # A few representative inputs: known model + benchmark, unknown model,
        # missing condition. predict() must always return a finite float in [0, 1].
        sample_inputs = [
            {
                'benchmark': 'mmlupro',
                'condition': 'zero-shot',
                'subject_content': 'Name: gpt-4o\nOrganization: OpenAI\nFamily: gpt-4o',
                'item_content': 'What is 2 + 2?',
            },
            {
                'benchmark': 'ai2d_test',
                'condition': 'none',
                'subject_content': 'Name: claude-3-5-sonnet-20240620',
                'item_content': 'Describe the diagram.',
            },
            {
                'benchmark': 'gsm8k',
                'condition': '',
                'subject_content': 'Name: __nonexistent_model_for_unk_test__',
                'item_content': 'A train leaves Boston ...',
            },
            {
                'benchmark': '__unknown_benchmark__',
                'condition': 'none',
                'subject_content': '',
                'item_content': '',
            },
        ]
        print('\nTrying predict() on sample inputs:')
        t_pred = time.perf_counter()
        for i, x in enumerate(sample_inputs, start=1):
            p = model_mod.predict(x, labeled=None)
            ok = isinstance(p, float) and math.isfinite(p) and 0.0 <= p <= 1.0
            status = 'OK' if ok else 'FAIL'
            print(f'  [{status}] sample {i}: bench={x["benchmark"]!r:20s} '
                  f'cond={x["condition"]!r:14s} subj={x["subject_content"][:40]!r:42s} -> p={p:.4f}')
            assert ok, f'predict() returned invalid value for sample {i}: {p!r}'
        pred_dt = time.perf_counter() - t_pred
        print(f'\n4 calls in {pred_dt*1000:.1f} ms total ({pred_dt*1000/4:.2f} ms/call avg, includes cache misses)')
    finally:
        sys.path.remove(str(extract_dir))
        for mod_name in ('model', 'latent_factor_pytorch'):
            sys.modules.pop(mod_name, None)

print('\n=== Bundle is upload-ready. ===')
print(f'Upload this file to Codabench: {CODABENCH_ZIP}')
print(f'  size: {CODABENCH_ZIP.stat().st_size / 1e6:.2f} MB')